In [0]:
%pip install pymupdf pillow databricks-vectorsearch mlflow pytesseract
dbutils.library.restartPython()

In [0]:
env = dbutils.notebook.run("/Workspace/anihan_tech_np/anihan_tech_np/00_Parameters/env_dependencies", 120)
print(env)

In [0]:
import os
import fitz
import io
from PIL import Image
import pytesseract
import uuid
import mlflow.deployments
from databricks.vector_search.client import (
    VectorSearchClient
)

In [0]:
df = spark.sql(f"select appcode, company_name, owner from anihan_tech_{env}.00_master_01_brz.masterdata_company")
display(df)

In [0]:
appcodes = [row["appcode"] for row in df.select("appcode").distinct().collect()]

dbutils.widgets.dropdown(
    "appcode",
    appcodes[0],      
    appcodes,
    "appcode"
)

selected_appcode = dbutils.widgets.get("appcode")
print(selected_appcode)

In [0]:
CATALOG = f"anihan_tech_{env}"
SCHEMA = f"00_{selected_appcode}_01_brz"
VOLUME_PATH = f"/Volumes/anihan_tech_{env}/00_{selected_appcode}_00_lz/lz_vl/pdf"
CHUNK_TABLE = f"{CATALOG}.{SCHEMA}.pdf_chunks"
VECTOR_ENDPOINT = f"{selected_appcode}-pdf-vector-endpoint"
VECTOR_INDEX = f"{CATALOG}.{SCHEMA}.pdf_chunks_index"
EMBEDDING_MODEL = "databricks-bge-large-en"
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
print(CATALOG)
print(SCHEMA)
print(VOLUME_PATH)
print(CHUNK_TABLE)
print(VECTOR_ENDPOINT)
print(VECTOR_INDEX)
print(EMBEDDING_MODEL)
print(LLM_MODEL)

In [0]:
def extract_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text_parts = []
    for page in doc:
        page_text = page.get_text()
        if page_text:
            text_parts.append(page_text)
        for img in page.get_images(full=True):
            try:
                xref = img[0]
                image_bytes = (
                    doc.extract_image(xref)["image"]
                )
                image = Image.open(
                    io.BytesIO(image_bytes)
                )
                ocr_text = (
                    pytesseract.image_to_string(image)
                )
                if ocr_text.strip():
                    text_parts.append(
                        f"""
[IMAGE_CONTENT]
{ocr_text}
[/IMAGE_CONTENT]
"""
                    )
            except Exception as e:
                print(
                    f"OCR failed for image in {pdf_path}: {e}"
                )
    return "\n".join(text_parts)

In [0]:
documents = []

for file in os.listdir(VOLUME_PATH):
    if not file.endswith(".pdf"):
        continue
    try:
        path = os.path.join(VOLUME_PATH, file)
        documents.append({
            "source": file,
            "text": extract_pdf(path)
        })
        print(f"Processed: {file}")
    except Exception as e:
        print(
            f"Failed processing {file}: {e}"
        )
print(
    f"Loaded {len(documents)} PDF documents"
)

In [0]:
def chunk_text(
    text,
    chunk_size=1000,
    overlap=200
):

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(
            text[start:end]
        )
        start += (
            chunk_size - overlap
        )
    return chunks

In [0]:
records = []
for doc in documents:
    chunks = chunk_text(doc["text"])
    for chunk_idx, chunk in enumerate(chunks):
        records.append({
            "id": str(uuid.uuid4()),
            "source": doc["source"],
            "chunk_id": chunk_idx,
            "chunk_text": chunk
        })
print(
    f"Created {len(records)} chunks"
)

In [0]:
chunk_df = spark.createDataFrame(records)

(
    chunk_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(CHUNK_TABLE)
)

In [0]:
client = (
    mlflow.deployments
    .get_deploy_client("databricks")
)

In [0]:
def get_embedding(text):

    response = client.predict(
        endpoint=EMBEDDING_MODEL,
        inputs={
            "input": [text]
        }
    )

    return (
        response.data[0]["embedding"]
    )

In [0]:
pdf_df = spark.table(CHUNK_TABLE)
rows = pdf_df.collect()
embedded_records = []

for row in rows:
    embedding = get_embedding(
        row["chunk_text"]
    )
    embedded_records.append({
        "id": row["id"],
        "source": row["source"],
        "chunk_id": row["chunk_id"],
        "chunk_text": row["chunk_text"],
        "embedding": embedding
    })

In [0]:
embedding_df = spark.createDataFrame(
    embedded_records
)

(
    embedding_df.write
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(CHUNK_TABLE)
)

In [0]:
vsc = VectorSearchClient()

In [0]:
vsc.create_endpoint(
    name=VECTOR_ENDPOINT,
    endpoint_type="STANDARD"
)

In [0]:
spark.sql(f"""
          alter table {CHUNK_TABLE}
          set tblproperties (
              delta.enableChangeDataFeed = true
          )
          """)

print(f"tblproperties for {CHUNK_TABLE} changed to true")

In [0]:
vsc.create_delta_sync_index(
    endpoint_name=VECTOR_ENDPOINT,
    index_name=VECTOR_INDEX,
    source_table_name=CHUNK_TABLE,
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_dimension=1024,
    embedding_vector_column="embedding"
)

In [0]:
#run this only after the pdf_chunks_index is provisioned

index = vsc.get_index(
    endpoint_name=VECTOR_ENDPOINT,
    index_name=VECTOR_INDEX
)
index.sync()

In [0]:
def retrieve_context(
    question,
    k=5
):
    query_embedding = (
        get_embedding(question)
    )
    results = (
        index.similarity_search(
            query_vector=query_embedding,
            columns=[
                "source",
                "chunk_text"
            ],
            num_results=k
        )
    )

    return results

In [0]:
def build_context(question):
    results = retrieve_context(
        question
    )
    chunks = []
    for row in (
        results["result"]["data_array"]
    ):
        chunks.append(row[1])
    return "\n\n".join(chunks)

In [0]:
def ask_pdf(question):
    context = build_context(
        question
    )
    prompt = f"""
You are an expert research analyst.
Use the provided study content to answer.

Requirements:
- Base answers only on the study.
- Cite findings when possible.
- Summarize complex information.
- Explain implications and recommendations.
- Do not hallucinate information.
- Do not make assumptions.
- Do not make up answers.
- Do not make up names, dates, numbers, organizations, products, or URLs.
- Do not make up sources.

Context:
{context}

Question:
{question}
"""
    response = client.predict(
        endpoint=LLM_MODEL,
        inputs={
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    return (
        response["choices"][0]
        ["message"]["content"]
    )

In [0]:
#test
response = ask_pdf(
    "What are some general question that can be asked from these pdf files?"
)

print(response)